Data Ingestion

In [1]:
import sys
print(sys.executable)

e:\Smart-Docs-Assistant\.venv\Scripts\python.exe


In [2]:
from langchain_core.documents import Document

In [3]:

from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyPDFLoader , PyMuPDFLoader, UnstructuredPDFLoader, UnstructuredWordDocumentLoader, UnstructuredPowerPointLoader, UnstructuredMarkdownLoader, UnstructuredHTMLLoader, UnstructuredEPubLoader, UnstructuredODTLoader, UnstructuredCSVLoader, UnstructuredExcelLoader

dir_loader = DirectoryLoader(
    "../data/pdf", 
    glob="**/*.pdf", 
    loader_cls=PyMuPDFLoader,
    show_progress=False
)

pdf_documents = dir_loader.load()
pdf_documents

C:\Users\ajilj\AppData\Local\Temp\ipykernel_21604\3831685698.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader
e:\Smart-Docs-Assistant\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-07-14T03:43:08+00:00', 'source': '..\\data\\pdf\\it_security_manual.pdf', 'file_path': '..\\data\\pdf\\it_security_manual.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-07-14T03:43:08+00:00', 'trapped': '', 'modDate': "D:20260714034308+00'00'", 'creationDate': "D:20260714034308+00'00'", 'page': 0}, page_content='IT Security Manual — Password & Access Policy\n1. Password Requirements\nAll employee passwords must be at least 14 characters long and include a mix of uppercase letters,\nlowercase letters, numbers, and symbols. Passwords must be changed every 90 days and may not reuse\nany of the previous 10 passwords.\n2. Multi-Factor Authentication\nMulti-factor authentication (MFA) is mandatory for all accounts with access to production systems,\ncustomer data, o

RAG Pipelines

In [4]:
import os
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyPDFLoader , PyMuPDFLoader, UnstructuredPDFLoader, UnstructuredWordDocumentLoader, UnstructuredPowerPointLoader, UnstructuredMarkdownLoader, UnstructuredHTMLLoader, UnstructuredEPubLoader, UnstructuredODTLoader, UnstructuredCSVLoader, UnstructuredExcelLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

Load Documents

In [5]:
from langchain_community.document_loaders import (
    PyMuPDFLoader,
    TextLoader,
    UnstructuredWordDocumentLoader,
    UnstructuredPowerPointLoader,
    UnstructuredMarkdownLoader,
    UnstructuredHTMLLoader,
    UnstructuredEPubLoader,
    UnstructuredODTLoader,
    UnstructuredCSVLoader,
    UnstructuredExcelLoader,
)

LOADERS = {
    "pdf": (PyMuPDFLoader, "**/*.pdf"),
    "text": (TextLoader, "**/*.txt"),
    "word": (UnstructuredWordDocumentLoader, "**/*.docx"),
    "ppt": (UnstructuredPowerPointLoader, "**/*.pptx"),
    "markdown": (UnstructuredMarkdownLoader, "**/*.md"),
    "html": (UnstructuredHTMLLoader, "**/*.html"),
    "epub": (UnstructuredEPubLoader, "**/*.epub"),
    "odt": (UnstructuredODTLoader, "**/*.odt"),
    "csv": (UnstructuredCSVLoader, "**/*.csv"),
    "excel": (UnstructuredExcelLoader, "**/*.xlsx"),
}

In [6]:
from pathlib import Path

def load_documents_from_directory(directory_path, file_types):
    all_documents = []

    for file_type in file_types:
        if file_type not in LOADERS:
            print(f"Unsupported file type: {file_type}")
            continue

        loader_cls, pattern = LOADERS[file_type]

        loader = DirectoryLoader(
            directory_path,
            glob=pattern,
            loader_cls=loader_cls,
            show_progress=True,
        )

        documents = loader.load()

        print(f"{file_type}: {len(documents)} file(s) loaded")

        for doc in documents:
            file_path = Path(doc.metadata.get("source", ""))

            doc.metadata.update({
                "source_file": file_path.name,
                "source_folder": file_path.parent.name,
                "file_type": file_type,
                "extension": file_path.suffix,
                "document_name": file_path.stem,
            })

        all_documents.extend(documents)

    print(f"\nTotal Documents Loaded: {len(all_documents)}")

    return all_documents

In [7]:
documents = load_documents_from_directory(
    "../data",
    [
        "pdf",
        "text",
        "word",
        "ppt",
        "markdown",
        "html",
        "epub",
        "odt",
        "csv",
        "excel"
    ]
)

100%|██████████| 3/3 [00:00<00:00, 280.66it/s]


pdf: 3 file(s) loaded


100%|██████████| 3/3 [00:00<00:00, 2954.43it/s]


text: 3 file(s) loaded


0it [00:00, ?it/s]


...

odt: 0 file(s) loaded


0it [00:00, ?it/s]


csv: 0 file(s) loaded


0it [00:00, ?it/s]

excel: 0 file(s) loaded

Total Documents Loaded: 6


In [8]:
print(documents[0].metadata)

{'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-07-14T03:43:08+00:00', 'source': '..\\data\\pdf\\it_security_manual.pdf', 'file_path': '..\\data\\pdf\\it_security_manual.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-07-14T03:43:08+00:00', 'trapped': '', 'modDate': "D:20260714034308+00'00'", 'creationDate': "D:20260714034308+00'00'", 'page': 0, 'source_file': 'it_security_manual.pdf', 'source_folder': 'pdf', 'file_type': 'pdf', 'extension': '.pdf', 'document_name': 'it_security_manual'}


Splitting

In [9]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = []
    for doc in documents:
        chunks = text_splitter.split_text(doc.page_content)
        for i, chunk in enumerate(chunks):
            new_doc = Document(
                page_content=chunk,
                metadata={**doc.metadata, "chunk_index": i}
            )
            split_docs.append(new_doc)

    return split_docs

In [10]:
chunks = split_documents(documents, chunk_size=1000, chunk_overlap=200)

In [11]:
chunks

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-07-14T03:43:08+00:00', 'source': '..\\data\\pdf\\it_security_manual.pdf', 'file_path': '..\\data\\pdf\\it_security_manual.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-07-14T03:43:08+00:00', 'trapped': '', 'modDate': "D:20260714034308+00'00'", 'creationDate': "D:20260714034308+00'00'", 'page': 0, 'source_file': 'it_security_manual.pdf', 'source_folder': 'pdf', 'file_type': 'pdf', 'extension': '.pdf', 'document_name': 'it_security_manual', 'chunk_index': 0}, page_content='IT Security Manual — Password & Access Policy\n1. Password Requirements\nAll employee passwords must be at least 14 characters long and include a mix of uppercase letters,\nlowercase letters, numbers, and symbols. Passwords must be changed every 90 days and may not reuse\nany of the previous 

Embeddings and Vector Store

In [12]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [13]:
class EmbeddingManager:
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)
        print(f"Embedding model '{model_name}' loaded successfully.")
        print(f"model dimensions: {self.model.get_embedding_dimension()}")
    
    def embed_texts(self, texts: List[str]) -> np.ndarray:
        return self.model.encode(texts, convert_to_numpy=True)

embedding_manager = EmbeddingManager()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6669.34it/s]


Embedding model 'all-MiniLM-L6-v2' loaded successfully.
model dimensions: 384


Vector Store

In [14]:
class VectorStore:
    
    def __init__(self, persist_directory: str = "../data/vector_store"):
    
        self.client = chromadb.PersistentClient(
            path=persist_directory
        )

        self.collection = self.client.get_or_create_collection(
            name="documents"
        )

        print(f"Vector store initialized at '{persist_directory}'.")
    def add_documents(self, documents: List[Document]):
        texts = [doc.page_content for doc in documents]
        metadatas = [doc.metadata for doc in documents]
        embeddings = embedding_manager.embed_texts(texts)
        
        ids = [str(uuid.uuid4()) for _ in range(len(documents))]
        
        self.collection.add(
            ids=ids,
            embeddings=embeddings.tolist(),
            metadatas=metadatas,
            documents=texts
        )
        print(f"Added {len(documents)} documents to the vector store.")
        print(f"Current number of documents in the collection: {self.collection.count()}")
        print(f"Embedding dimension: {embeddings.shape[1]}")
        print(f"Sample embedding for the first document: {embeddings[0][:5]}...")  # Print first 5 dimensions of the first embedding
    def query(self, query_text: str, top_k: int = 5) -> List[Dict[str, Any]]:
        query_embedding = embedding_manager.embed_texts([query_text])[0]
        
        results = self.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=top_k
        )
        
        return [
            {
                "document": doc,
                "metadata": metadata,
                "score": score
            }
            for doc, metadata, score in zip(results['documents'][0], results['metadatas'][0], results['distances'][0])
        ]

In [15]:
vectorstore = VectorStore()

Vector store initialized at '../data/vector_store'.


Convert text to embeddings

In [16]:
texts = [doc.page_content for doc in chunks]

embeddings = embedding_manager.embed_texts(texts)

vectorstore.add_documents(chunks)

Added 13 documents to the vector store.
Current number of documents in the collection: 39
Embedding dimension: 384
Sample embedding for the first document: [-0.0138096   0.01545463 -0.05885087 -0.04664924  0.00876228]...


Retriever pipeline

In [17]:
class Retriever:
    
    def __init__(self, vector_store: VectorStore):
        self.vector_store = vector_store
    
    def retrieve(self, query: str, top_k: int = 5) -> List[Dict[str, Any]]:
        return self.vector_store.query(query, top_k=top_k)

In [18]:
retriever = Retriever(vectorstore)

In [19]:
retriever.retrieve("what are the  Parental Leave")

[{'document': "Company Leave Policy\n1. Annual Leave\nAll full-time employees are entitled to 21 days of paid annual leave per calendar year. Leave accrues\nmonthly at a rate of 1.75 days per month. Unused leave may be carried over to the next year up to a\nmaximum of 5 days; any excess is forfeited on December 31st.\n2. Sick Leave\nEmployees receive 10 days of paid sick leave per year. A medical certificate is required for sick leave\nexceeding 2 consecutive days. Sick leave does not carry over to the next year.\n3. Parental Leave\nPrimary caregivers are entitled to 16 weeks of paid parental leave. Secondary caregivers are entitled to 4\nweeks of paid parental leave. Parental leave must be taken within 12 months of the child's birth or adoption\ndate.\n4. Requesting Leave\nLeave requests must be submitted through the HR portal at least 5 business days in advance for planned\nleave. Emergency leave should be reported to your manager as soon as possible, with formal",
  'metadata': {'to

LLM Integration

In [21]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()  

groq_api_key = os.getenv("GROQ_API_KEY")

In [24]:
llm = ChatGroq(
    api_key=groq_api_key,
    model="llama-3.3-70b-versatile",
    temperature=0.1,
    max_tokens=512
)



In [25]:
response = llm.invoke("Hello, can you answer questions about documents?")
print(response.content)

I'd be happy to help answer your questions about documents. What kind of documents are you referring to (e.g. legal, business, historical, etc.) and what would you like to know?


In [26]:
from langchain_core.prompts import ChatPromptTemplate

rag_prompt = ChatPromptTemplate.from_template("""
You are a helpful document assistant.

Answer the question using only the context below.
If the answer is not in the context, say: "I don't know based on the provided documents."

Context:
{context}

Question:
{question}

Answer:
""")

In [27]:
def format_context(retrieved_docs):
    return "\n\n".join(
        [
            f"Source: {doc['metadata'].get('source_file', 'unknown')}\n{doc['document']}"
            for doc in retrieved_docs
        ]
    )

In [28]:
def answer_question(question, top_k=5):
    retrieved_docs = retriever.retrieve(question, top_k=top_k)
    context = format_context(retrieved_docs)

    prompt = rag_prompt.format_messages(
        context=context,
        question=question
    )

    response = llm.invoke(prompt)

    return {
        "answer": response.content,
        "sources": [
            doc["metadata"].get("source_file", "unknown")
            for doc in retrieved_docs
        ]
    }

In [29]:
result = answer_question("What are the parental leave policies?")
print(result["answer"])
print("Sources:", result["sources"])

Primary caregivers are entitled to 16 weeks of paid parental leave, while secondary caregivers are entitled to 4 weeks of paid parental leave. Parental leave must be taken within 12 months of the child's birth or adoption date.
Sources: ['leave_policy.pdf', 'leave_policy.pdf', 'leave_policy.pdf', 'leave_policy.txt', 'leave_policy.txt']
